This notebook demonstrates two examples
1. Create the embeddings for IMDB data using TF-IDF vectorizer and store the embeddings in chromaDB. Query the chromadb for similar reviews

2. 


In [ ]:
#pip install chromadb

In [ ]:
import chromadb

In [ ]:
chromadb.__version__

--------------------------
#### ChromDB

- add TF-IDF vectors into ChromaDB
- Query the database
----------------------------

In [ ]:
import pandas as pd

Use **IMDB** dataset

In [ ]:
# Load the IMDb dataset
file_path = r'D:\Makesh\Working\AI\RPS\Day06-22 Nov\Datasets\IMDB-cleaned-text.csv'
df = pd.read_csv(file_path)

In [ ]:
df.shape

In [ ]:
# Take only the first 1000 reviews
reviews = df['review'].sample(1000).tolist()

#### Generate TF-IDF Vectors
- Use scikit-learn's TfidfVectorizer to generate TF-IDF vectors for the movie reviews.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

Initialize the TF-IDF vectorizer
max_features=1000 ==> This tells the vectorizer to keep only the top 1000 important words #(based on frequency across the dataset).
max_features=1000 applies to the entire dataset, not each document
# Meaning:
TF-IDF builds a vocabulary of size 1000
This vocabulary is shared across all documents
Every document is represented using these same 1000 features

In [ ]:
vectorizer = TfidfVectorizer(max_features=1000)  # Limiting to 1000 features for efficiency

In [ ]:
# Generate TF-IDF vectors for the reviews. This retuns the sparse matrix
tfidf_matrix = vectorizer.fit_transform(reviews)


In [ ]:
tfidf_matrix.shape

In [ ]:
import chromadb

from chromadb.config import Settings

In [ ]:
# Initialize ChromaDB client
client = chromadb.Client(Settings(allow_reset = True))

In [ ]:
# List all collections
collections = client.list_collections()
print([collection.name for collection in collections])

In [ ]:
# Create a collection to store TF-IDF vectors
collection_name = client.get_or_create_collection("imdb_reviews")

In [ ]:
collection_name.count()

In [ ]:
# Attempt to delete the collection
# try:
#     client.delete_collection(name=collection_name)  # Pass the name as a keyword argument
#     print(f"Collection '{collection_name}' deleted successfully.")
# except Exception as e:
#     print(f"Error deleting collection: {e}")

In [ ]:
# Convert TF-IDF spare matrix to dense array and insert into ChromaDB. This is because ChromaDB requires dense vectors.
tfidf_dense = tfidf_matrix.toarray()


In [ ]:
# Add each review vector into ChromaDB
# takes abt 1 min
for idx, vector in enumerate(tfidf_dense):
    collection_name.add(
        ids       =[str(idx)],                  # Unique ID for each review
        embeddings=[vector],                    # The TF-IDF vector. ChromaDB expects a list of vectors even for single entries
        metadatas =[{"review": reviews[idx]}],  # Store the actual review
    )

- ids: Unique identifier for each review.
- embeddings: The TF-IDF vectors.
- metadatas: Metadata like the actual review text, which will be retrieved.

#### Querying ChromaDB with TF-IDF
- query ChromaDB to retrieve similar reviews using a TF-IDF-based retriever.

In [ ]:
def query_chromadb(query_text, top_k=5):
    # Convert the query to a TF-IDF vector
    query_vector = vectorizer.transform([query_text]).toarray()[0]
    
    # Perform similarity search in ChromaDB
    results = collection_name.query(
        query_embeddings=[query_vector],  # The query vector
        n_results       =top_k  # Number of results to return
    )
    
    return results

In [ ]:
# Example query
query_text = "I love movies about space adventures"
result     = query_chromadb(query_text)

In [ ]:
type(result)

In [ ]:
result.keys()

In [ ]:
result['ids']

In [ ]:
result['distances']

In [ ]:
result['metadatas']



In [ ]:
for idx, review in enumerate(result['metadatas'][0]):
    print(review['review'])
    print('---')

using **books CSV**

In [46]:
# Load the books csv
# https://www.kaggle.com/datasets/saurabhbagchi/books-dataset/data
file_path = r'D:\Makesh\Working\AI\RPS\Day06-22 Nov\Datasets\books_data\books.csv'
df = pd.read_csv(file_path, encoding='ISO-8859-1', sep=';', on_bad_lines='skip', low_memory=False)

In [47]:
df.shape

(271360, 8)

In [48]:
df.drop(['Image-URL-S', 'Image-URL-M',	'Image-URL-L'], axis=1, inplace=True)

In [49]:
df.sample(10)

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
219318,0451197380,Second Chance,Dan Montague,1999,Signet Book
97823,0786803002,The Birchbark House,Louise Erdrich,1999,Hyperion Books for Children
122488,0871132605,Kill the Poor,Joel Rose,1988,Pub Group West
89382,0671611089,LORD MOUNTBATTEN,David Butler,1986,Pocket
74862,0394540654,Great Detectives: A Century of the Best Myster...,David W. McCullough,1984,Random House Inc
122629,0804101590,"Suspicion (Park Avenue, No 3)",Lorayne Ashton,1987,Ivy Books
253578,0679454608,"The Rooms of Heaven: A Story of Love, Death, G...",Mary Allen,1999,Random House Inc
152445,0881341045,Apple II User's Guide,Lon Poole,1983,McGraw-Hill
116421,0671737759,BETHANY'S SIN,Robert R. McCammon,1988,Pocket
49665,0590405233,White Fang,Jack London,1986,Scholastic


In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271360 entries, 0 to 271359
Data columns (total 5 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   ISBN                 271360 non-null  object
 1   Book-Title           271360 non-null  object
 2   Book-Author          271358 non-null  object
 3   Year-Of-Publication  271360 non-null  object
 4   Publisher            271358 non-null  object
dtypes: object(5)
memory usage: 10.4+ MB


In [51]:
import string,re
#import spacy
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [52]:
def preprocess_text(text):
    #print(f"Original text: {text}")
    
    text = text.lower()  # Lowercasing
    #print(f"Lowercased text: {text}")
    
    # Remove all punctuation except '&'
    text = text.translate(str.maketrans('', '', string.punctuation.replace('&', '')))
    #print(f"Without punctuation (keeping '&'): {text}")
    
    text = text.strip()  # Remove leading/trailing whitespace
    text = re.sub(r'\s+', ' ', text)  # Remove excessive whitespace using regex
    #print(f"Without excessive whitespace: {text}")
    
    # Normalize &amp; if it exists
    text = re.sub(r'&amp;', 'and', text)
    #print(f"After replacing '&amp;': {text}")
    
    # Replace any remaining & with 'and'
    text = text.replace('&', 'and')
    #print(f"After replacing '&': {text}")
    
    return text

In [53]:
# Example usage
sample_text = "This is an example   with   excessive  & whitespace!"
cleaned_text = preprocess_text(sample_text)
print(cleaned_text)

this is an example with excessive and whitespace


In [54]:
# Drop rows with any null values
df_cleaned = df.dropna()

In [55]:
%%time
# Apply the preprocessing
df_cleaned['text'] = df_cleaned['Book-Title'] + ' ' + df_cleaned['Book-Author'] + ' ' + df_cleaned['Publisher']
df_cleaned['text'] = df_cleaned['text'].apply(preprocess_text)

<timed exec>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


CPU times: total: 1.77 s
Wall time: 1.76 s


<timed exec>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [56]:
df_cleaned.shape

(271356, 6)

In [57]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 271356 entries, 0 to 271359
Data columns (total 6 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   ISBN                 271356 non-null  object
 1   Book-Title           271356 non-null  object
 2   Book-Author          271356 non-null  object
 3   Year-Of-Publication  271356 non-null  object
 4   Publisher            271356 non-null  object
 5   text                 271356 non-null  object
dtypes: object(6)
memory usage: 14.5+ MB


In [ ]:
# Sample 2500 records
df_cleaned_samples = df_cleaned.sample(2500)

In [59]:
# Initialize the TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=10000)

In [60]:
# Generate TF-IDF vectors for the reviews
tfidf_matrix = vectorizer.fit_transform(df_cleaned_samples.text)

In [ ]:
# Reset the client, which clears all collections and data
# Reusing the existing client
client.reset()

print("ChromaDB client has been reset.")

ChromaDB client has been reset.


In [62]:
# List all collections
collections = client.list_collections()
print([collection.name for collection in collections])

[]


In [63]:
# Create a collection to store TF-IDF vectors
collection_name = client.get_or_create_collection("book_info")

In [64]:
# Delete the collection
# client.delete_collection(collection_name)

# print(f"Collection '{collection_name}' has been deleted.")

In [65]:
# Convert TF-IDF matrix to dense array and insert into ChromaDB
tfidf_dense = tfidf_matrix.toarray()

In [66]:
tfidf_dense.shape

(2500, 8599)

In [67]:
df_cleaned_samples.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'text'],
      dtype='object')

In [69]:
%%time

# takes abt 1-3 mins
# Store the original indices before any resetting
original_indices = df_cleaned_samples.index.tolist()  # Store original indices

# Add each review vector into ChromaDB
for idx, vector in enumerate(tfidf_dense):

    # Use original_indices to fetch metadata
    original_idx = original_indices[idx]
    
    # Construct metadata with book information
    metadata = {
        "Title": df_cleaned_samples.loc[original_idx, 'Book-Title'],          # Book Title
        "Author": df_cleaned_samples.loc[original_idx, 'Book-Author'],        # Book Author
        "Year": df_cleaned_samples.loc[original_idx, 'Year-Of-Publication'],  # Year of Publication
        "Publisher": df_cleaned_samples.loc[original_idx, 'Publisher'],       # Publisher
    }
    
    # Add the vector and metadata to ChromaDB collection
    collection_name.add(
        ids=[str(idx)],                  # Unique ID for each book/review
        embeddings=[vector],             # The TF-IDF vector
        metadatas=[metadata]             # Store book details (metadata)
    )

CPU times: total: 35.7 s
Wall time: 24.2 s


#### Querying ChromaDB with TF-IDF
- query ChromaDB to retrieve similar reviews using a TF-IDF-based retriever.

In [68]:
def query_chromadb(query_text, top_k=3):
    # Convert the query to a TF-IDF vector
    query_vector = vectorizer.transform([query_text]).toarray()[0]
    
    # Perform similarity search in ChromaDB
    results = collection_name.query(
        query_embeddings=[query_vector],  # The query vector
        n_results       =top_k            # Number of results to return
    )
    
    return results

In [70]:
df_cleaned_samples.sample(10)

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,text
190434,0297607391,The rainbow singer,Simon Kerr,2001,Weidenfeld &amp; Nicolson,the rainbow singer simon kerr weidenfeld andam...
192977,1880241129,Understanding Bats,Bird Watchers Digest Pr,1996,Bird Watchers Digest Pr,understanding bats bird watchers digest pr bir...
204015,1583143858,Could It Be Magic? (Arabesque),Deirdre Savoy,2003,Bet Books,could it be magic arabesque deirdre savoy bet ...
126300,031232149X,A Hard Ticket Home,David Housewright,2004,St. Martin's Minotaur,a hard ticket home david housewright st martin...
141064,0425043045,Trader to the Stars,Poul Anderson,1979,Penguin Putnam~mass,trader to the stars poul anderson penguin putn...
239574,1857038312,CVS for Graduates,Gerald Higginbottom,2002,Parkwest Publications,cvs for graduates gerald higginbottom parkwest...
55387,3499330709,Tatort Finanzministerium: Die staatlichen Helf...,Joachim Wagner,1986,Rowohlt,tatort finanzministerium die staatlichen helfe...
65554,1565072294,Who Brings Forth the Wind (Kensington Chronicles),Lori Wick,1994,Harvest House Publishers,who brings forth the wind kensington chronicle...
169958,0553445480,"Untamed (Loveswept, No 785)",Cynthia Powell,1996,Loveswept,untamed loveswept no 785 cynthia powell loveswept
231344,0380751232,The Rainbow Cadenza: A Novel in Vistata Form,J. Neil Schulman,1986,Harper Mass Market Paperbacks (Mm),the rainbow cadenza a novel in vistata form j ...


In [72]:
# Example query
query_text = "Singer"
result = query_chromadb(query_text)

In [73]:
type(result)

dict

In [74]:
result.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

In [75]:
result['ids']

[['165', '624', '1873']]

In [76]:
result['distances']

[[1.1900134086608887, 1.2228120565414429, 1.341149091720581]]

In [77]:
for idx, info in enumerate(result['metadatas'][0]):
    print(info['Author'])
    
    print(info['Title'])
    print(info['Publisher'])
    print(info['Year'])
    print('---')

Isaac Bashevis Singer
Shosha : A Novel
Noonday Press
1996
---
Simon Kerr
The rainbow singer
Weidenfeld &amp; Nicolson
2001
---
Israel Zamir
Journey to My Father, Isaac Bashevis Singer
Arcade Publishing
1995
---


#### Why we used ChromaDB (vector database)

`Efficient Storage of High-Dimensional Data`
- ChromaDB is optimized for storing and retrieving embeddings (like those produced by TF-IDF, word2vec, or other vectorization methods) that can be high-dimensional and sparse. This optimization helps in efficiently managing large datasets.

`Fast Similarity Search`
- It provides efficient querying capabilities to perform similarity searches. For instance, you can quickly find similar documents or items based on their vector representations, which is crucial in recommendation systems and search applications.

`Metadata Storage`
- Alongside the embeddings, ChromaDB allows for the storage of associated metadata, making it easier to retrieve relevant information about the embeddings (like book titles, authors, publication years, etc.) in addition to the embeddings themselves.

`Support for Various Embedding Types`
- ChromaDB can handle different types of embeddings, whether they are generated from TF-IDF, neural networks, or other methods, allowing flexibility in how you represent your data.

`Example Use Cases`
- **Recommendation Systems**: Finding similar books based on user preferences.
- **Search Engines**: Enabling fast searches for documents or products based on vector similarity.
- **Natural Language Processing Applications**: Enhancing tasks like semantic search, document clustering, and classification.


